In [1]:
from pathlib import Path

from mltrainer import Trainer, metrics
from mltrainer.rnn_models import NLPmodel, AttentionNLP

import torch
from torch.utils.data import DataLoader
from torch import optim


from mads_datasets import DatasetFactoryProvider, DatasetType

We load the streamers from the datasetfactory

In [2]:
imdbdatasetfactory = DatasetFactoryProvider.create_factory(DatasetType.IMDB)

In [3]:
datasets = imdbdatasetfactory.create_dataset()

2026-06-19 13:37:37.993 | INFO     | mads_datasets.base:download_data:94 - Start download...
  0%|          | 0.00/84.1M [00:00<?, ?iB/s]2026-06-19 13:37:38.615 | INFO     | mads_datasets.datatools:get_file:105 - Downloading /root/.cache/mads_datasets/imdb/aclImdb_v1.tar.gz
100%|██████████| 84.1M/84.1M [00:13<00:00, 6.34MiB/s]
2026-06-19 13:37:51.894 | INFO     | mads_datasets.datatools:extract:128 - Unzipping /root/.cache/mads_datasets/imdb/aclImdb_v1.tar.gz
2026-06-19 13:38:09.875 | INFO     | mads_datasets.base:download_data:112 - Digest of /root/.cache/mads_datasets/imdb/aclImdb_v1.tar.gz matches expected digest
2026-06-19 13:38:09.876 | INFO     | mads_datasets.base:download_data:117 - Removing unzipped file /root/.cache/mads_datasets/imdb/aclImdb_v1.tar.gz
2026-06-19 13:38:13.337 | INFO     | mads_datasets.factories.basicfactories:create_dataset:85 - Creating TextDatasets from 25000 trainfilesand 25000 testfiles.
100%|██████████| 25000/25000 [00:00<00:00, 27054.80it/s]


In [4]:
traindataset = datasets["train"]

In [5]:
imdbdatasetfactory.settings

dataset_url: https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz
filename: aclImdb_v1.tar.gz
name: imdb
unzip: True
formats: [<FileTypes.TXT: '.txt'>]
digest: 7c2ac02c03563afcf9b574c7e56c153a
maxvocab: 10000
maxtokens: 100

In [6]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.normalizers import Lowercase, StripAccents, Sequence, NFD, Replace

In [7]:
tokenizer = Tokenizer(BPE())
trainer = BpeTrainer(special_tokens=["<unk>"], vocab_size=10000)
tokenizer.pre_tokenizer = Whitespace()
normalizer = Sequence([NFD(), Replace("<br />", ""), StripAccents(), Lowercase()])
tokenizer.normalizer = normalizer
tokenizer.train_from_iterator(traindataset, trainer=trainer)
print(f"the vocab size is {tokenizer.get_vocab_size()}")

the vocab size is 10000


In [8]:
?BpeTrainer

In [9]:
tokenizer.get_vocab()

{'discer': 9415,
 'reliable': 9537,
 'complain': 2812,
 'cand': 4236,
 'ite': 322,
 'status': 5253,
 'gor': 2540,
 'hitting': 6236,
 'less': 468,
 'femme': 8754,
 'natal': 8526,
 'nail': 7172,
 'jeffrey': 7263,
 'bye': 6429,
 'slice': 8732,
 'racing': 9752,
 'olph': 9806,
 '1977': 9928,
 'nis': 4552,
 'reminiscent': 5391,
 '??': 1527,
 'anywhere': 3811,
 'mine': 3364,
 'hoo': 5842,
 'version': 1007,
 'contrad': 7527,
 'bring': 1601,
 'pops': 7711,
 'och': 9315,
 'tops': 9193,
 'lum': 4224,
 'immortal': 9383,
 'brosn': 6500,
 'adle': 7520,
 'surround': 3244,
 'guaranteed': 9300,
 'resp': 2165,
 'marketing': 8493,
 'resid': 6697,
 'chan': 2747,
 'tense': 5835,
 'writing': 1430,
 'yet': 870,
 'invol': 1095,
 'disgusting': 4652,
 'ader': 5574,
 'ating': 595,
 'depressing': 4589,
 'val': 823,
 'rosemary': 9168,
 'haired': 8984,
 'unexplained': 9408,
 'humour': 2919,
 'fleet': 9673,
 'included': 4083,
 'street': 2161,
 'kept': 2013,
 'issu': 2099,
 'hidden': 3537,
 'flaw': 2046,
 'saves': 61

In [10]:
from torch.nn.utils.rnn import pad_sequence
import torch

Tensor = torch.Tensor


class Preprocessor:
    def __init__(
        self, max: int, tokenizer
    ) -> None:
        self.max = max
        self.tokenizer = tokenizer

    def cast_label(self, label: str) -> int:
        if label == "neg":
            return 0
        else:
            return 1

    def __call__(self, batch: list) -> tuple[Tensor, Tensor]:
        labels, text = [], []
        for x, y in batch:
            tokens = torch.tensor(self.tokenizer.encode(x).ids)
            tokens = tokens[:self.max]
            text.append(tokens)
            labels.append(self.cast_label(y))

        text_ = pad_sequence(text, batch_first=True, padding_value=0)
        return text_, torch.tensor(labels)


In [11]:
preprocessor = Preprocessor(256, tokenizer)
streamers = imdbdatasetfactory.create_datastreamer(batchsize=32, preprocessor=preprocessor)

2026-06-19 13:38:24.558 | INFO     | mads_datasets.base:download_data:121 - Folder already exists at /root/.cache/mads_datasets/imdb
2026-06-19 13:38:27.250 | INFO     | mads_datasets.factories.basicfactories:create_dataset:85 - Creating TextDatasets from 25000 trainfilesand 25000 testfiles.
100%|██████████| 25000/25000 [00:00<00:00, 42804.09it/s]


In [12]:
train = streamers["train"]
batch = train.batchloop()
batch

[("The combination of the superb black and white photography and the 'Eugene Onegin with a twist' plot made this a real knock out for me. The atmosphere created by the mostly very dark shots contrasted with occasional very bright overexposed white was gripping. There was a superb moment where where transparencies - apparently conventional holiday snaps but where the faces of the actors revealed character and situation subtly but instantly - were shown accompanied by Lensky's heart-wrenching aria from the Tschaikowsky opera Eugene Onegin.<br /><br />For me the mark of a good film is that it should take advantage of the opportunities presented by that medium, which means that often the story is less important than imagery and atmosphere - Last Year in Marienbad is a good example of such a film. Krisana is in the same mould.",
  'pos'),
 ('A group of hunters track down a werewolf, kill it, decapitate it and then sell the head to unethical Dr. Atwill (played by director/writer Tim Sullivan

In [13]:
train = streamers["train"]
print(f"number of batches {len(train)}")
trainstreamer = train.stream()
validstreamer = streamers["valid"].stream()
X, y = next(iter(trainstreamer))
X.shape, y.shape

number of batches 781


(torch.Size([32, 256]), torch.Size([32]))

In [14]:
X

tensor([[ 845,  347, 2054,  ...,    9,   59,  318],
        [3750, 4718,  960,  ..., 1975,  129, 2863],
        [5659, 5158,   52,  ...,    0,    0,    0],
        ...,
        [  49, 1332,  122,  ...,    0,    0,    0],
        [2231, 9914,    9,  ...,    0,    0,    0],
        [ 307,  180,  302,  ...,    0,    0,    0]])

The full dataset has 782 batches of 32 examples

Setup accuracy and loss_fn (this is a classification problem with two classes, 0 and 1)

In [15]:
accuracy = metrics.Accuracy()
loss_fn = torch.nn.CrossEntropyLoss()
log_dir = Path("logs/nlp/").resolve()
log_dir


PosixPath('/content/logs/nlp')

Basic config. We need to specify the vocabulary lenght for the embedding layer.
Trainsteps are set to just 100 batches for speedup in the demo.

In [16]:
from mltrainer import TrainerSettings, ReportTypes

settings = TrainerSettings(
    epochs=3,
    metrics=[accuracy],
    logdir=log_dir,
    train_steps=100,
    valid_steps=25,
    reporttypes=[ReportTypes.TENSORBOARD],
    scheduler_kwargs={"factor": 0.5, "patience": 5},
)
settings

2026-06-19 13:38:28.581 | INFO     | mltrainer.settings:check_path:60 - Created logdir /content/logs/nlp


epochs: 3
metrics: [Accuracy]
logdir: /content/logs/nlp
train_steps: 100
valid_steps: 25
reporttypes: [<ReportTypes.TENSORBOARD: 'TENSORBOARD'>]
optimizer_kwargs: {'lr': 0.001, 'weight_decay': 1e-05}
scheduler_kwargs: {'factor': 0.5, 'patience': 5}
earlystop_kwargs: {'save': False, 'verbose': True, 'patience': 10}

In [17]:
from mltrainer.rnn_models import ModelConfig
config = ModelConfig(
    input_size=tokenizer.get_vocab_size(),
    hidden_size=128,
    dropout=0.1,
    num_layers=1,
    output_size=2,
)
config

ModelConfig(input_size=10000, hidden_size=128, num_layers=1, output_size=2, dropout=0.1)

In [18]:
model = NLPmodel(config)
model

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/rnn.py:1364: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.1 and num_layers=1
  super().__init__("GRU", *args, **kwargs)


NLPmodel(
  (emb): Embedding(10000, 128)
  (rnn): GRU(128, 128, batch_first=True, dropout=0.1)
  (linear): Linear(in_features=128, out_features=2, bias=True)
)

The base NLP model is just a GRU, with an embedding as a first layer.


In [19]:
if torch.backends.mps.is_available() and torch.backends.mps.is_built():
    device = torch.device("mps")
    print("Using MPS")
elif torch.cuda.is_available():
    device = "cuda:0"
    print("using cuda")
else:
    device = "cpu"
    print("using cpu")

using cuda


In [20]:
optimizer = optim.Adam
scheduler = optim.lr_scheduler.ReduceLROnPlateau

trainer = Trainer(
    model=model,
    settings=settings,
    loss_fn=loss_fn,
    optimizer=optimizer,
    traindataloader=trainstreamer,
    validdataloader=validstreamer,
    scheduler=scheduler,
    device=device,
    )

2026-06-19 13:38:28.762 | INFO     | mltrainer.trainer:dir_add_timestamp:23 - Logging to /content/logs/nlp/20260619-133828
2026-06-19 13:38:35.149 | INFO     | mltrainer.trainer:__init__:65 - Found earlystop_kwargs in settings.Set to None if you dont want earlystopping.


In [21]:
trainer.loop()

100%|██████████| 100/100 [00:03<00:00, 26.89it/s]
2026-06-19 13:38:43.326 | INFO     | mltrainer.trainer:report:221 - Epoch 0 train 0.7044 test 0.6992 metric ['0.4863']
100%|██████████| 100/100 [00:02<00:00, 33.81it/s]
2026-06-19 13:38:46.825 | INFO     | mltrainer.trainer:report:221 - Epoch 1 train 0.7006 test 0.6956 metric ['0.5250']
100%|██████████| 100/100 [00:02<00:00, 38.01it/s]
2026-06-19 13:38:49.994 | INFO     | mltrainer.trainer:report:221 - Epoch 2 train 0.6953 test 0.6978 metric ['0.5025']
2026-06-19 13:38:49.995 | INFO     | mltrainer.trainer:__call__:264 - best loss: 0.6956, current loss 0.6978.Counter 1/10.
100%|██████████| 3/3 [00:11<00:00,  3.77s/it]


Compare the impact of attention

In [22]:
attentionmodel = AttentionNLP(config)

attentiontrainer = Trainer(
    model=attentionmodel,
    settings=settings,
    loss_fn=loss_fn,
    optimizer=optim.Adam,
    traindataloader=trainstreamer,
    validdataloader=validstreamer,
    scheduler=optim.lr_scheduler.ReduceLROnPlateau,
    device=device,
    )

attentiontrainer.loop()

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/rnn.py:1364: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.1 and num_layers=1
  super().__init__("GRU", *args, **kwargs)
2026-06-19 13:38:50.017 | INFO     | mltrainer.trainer:dir_add_timestamp:23 - Logging to /content/logs/nlp/20260619-133850
2026-06-19 13:38:50.021 | INFO     | mltrainer.trainer:__init__:65 - Found earlystop_kwargs in settings.Set to None if you dont want earlystopping.
100%|██████████| 100/100 [00:03<00:00, 29.27it/s]
2026-06-19 13:38:54.040 | INFO     | mltrainer.trainer:report:221 - Epoch 0 train 0.6729 test 0.5987 metric ['0.6913']
100%|██████████| 100/100 [00:04<00:00, 24.51it/s]
2026-06-19 13:38:58.713 | INFO     | mltrainer.trainer:report:221 - Epoch 1 train 0.5596 test 0.5219 metric ['0.7562']
100%|██████████| 100/100 [00:03<00:00, 29.54it/s]
2026-06-19 13:39:02.696 | INFO     | mltrainer.trainer:repo